# UniCell Ontology-Guided Annotation and Benchmarking

This notebook was converted from `docs/TUTORIALS/Tutorial_01_UniCell_Ontology_Guided_Annotation.md` and uses human liver data to demonstrate how to:

1. Train UniCell on a new dataset;
2. Perform ontology-guided cell-type annotation on a test set;
3. Compute model evaluation metrics;
4. Compare the results with existing predictions from scANVI, OnClass, CellTypist, and SingleR.

> The current version of UniCell jointly learns cell type, tissue, and species. Therefore, the training data's `obs` must contain label fields for all three tasks.

> Model training is disabled by default to avoid unintentionally starting a long-running task. To retrain the model, set `RUN_TRAINING = True` in the configuration.

## Data Download and Directory Structure

Example data download: <https://bgipan.genomics.cn/#/link/Z2z61owwv8WAqbuIcCdB>  
Access code: `YC3A`

After downloading the files, the following layout is recommended:

```text
data/
├── hsa_liver_train.h5ad
├── hsa_liver_eval.h5ad
└── hsa_liver_test.h5ad
```

If the files are stored elsewhere, only the paths in the configuration cell below need to be modified.

## Step 1: Load the Runtime Environment

In [ ]:
from pathlib import Path
import os
import pickle

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from unicell.anno_predict import unicell_predict
from unicell.scDataset import scDataset
from unicell.trainer import UnicellTrainer
from unicell.utils.utils import compute_metrics

try:
    from palettable.cartocolors.diverging import TealRose_5
    BENCHMARK_COLORS = TealRose_5.mpl_colors[::-1]
except ImportError:
    BENCHMARK_COLORS = list(plt.get_cmap("Set2").colors[:5])
    print("palettable is not installed; using the default Matplotlib color palette.")

print("Runtime environment loaded successfully")
print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

## Step 2: Configure the Data, Model, and Label Fields

Usually, only the paths need to be modified. `BASELINE_PREDICTIONS` contains prediction files already generated by the other four methods. If these files are not yet available, UniCell training and testing can be completed first, and the baseline comparison section can be skipped.

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

TRAIN_DATA = DATA_DIR / "hsa_liver_train.h5ad"
EVAL_DATA = DATA_DIR / "hsa_liver_eval.h5ad"
TEST_DATA = DATA_DIR / "hsa_liver_test.h5ad"
MODEL_DIR = PROJECT_DIR / "models/hsa_liver"
ANNOTATED_TEST_OUTPUT = PROJECT_DIR / "results/hsa_liver_unicell_annotated.h5ad"
BASELINE_PREDICTIONS = PROJECT_DIR / "runs/results/hsa_liver_preds.h5ad"
METRIC_FIGURE_OUTPUT = PROJECT_DIR / "results/hsa_liver_metric.png"

CELL_TYPE_KEY = "cell_type_ontology_term_id"
TISSUE_KEY = "general_tissue"
SPECIES_KEY = "organism"

DEVICE = "auto"
RUN_TRAINING = False       # Set to True only when the model needs to be retrained
TRAIN_BATCH_SIZE = 128
PREDICT_BATCH_SIZE = 512
NUM_EPOCHS = 30

if DEVICE == "auto":
    device = "cuda" if torch.cuda.is_available() else "cpu"
elif DEVICE in {"cuda", "cpu"}:
    device = DEVICE
else:
    raise ValueError('DEVICE must be "auto", "cuda", or "cpu".')

if device == "cuda" and not torch.cuda.is_available():
    print("No GPU is currently available; automatically switching to CPU.")
    device = "cpu"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATED_TEST_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
METRIC_FIGURE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

print(f"Device: {device}")
print(f"Retrain model: {RUN_TRAINING}")
print(f"Model directory: {MODEL_DIR}")

## Step 3: Check the Data Fields

This step reads only `obs` and the data shape for validation. The training set must contain cell type, tissue, and species; the test set must contain at least an expression matrix. Metrics can be computed if ground-truth cell-type labels are available.

In [ ]:
paths_to_check = {"train": TRAIN_DATA, "eval": EVAL_DATA, "test": TEST_DATA}
missing_files = [f"{name}: {path}" for name, path in paths_to_check.items() if not path.is_file()]
if missing_files:
    raise FileNotFoundError("The following data files were not found:\n- " + "\n- ".join(missing_files))

dataset_overview = []
for split_name, data_path in paths_to_check.items():
    backed_data = ad.read_h5ad(data_path, backed="r")
    required = (
        [CELL_TYPE_KEY, TISSUE_KEY, SPECIES_KEY]
        if split_name in {"train", "eval"}
        else []
    )
    missing_columns = [column for column in required if column not in backed_data.obs.columns]
    dataset_overview.append({
        "split": split_name,
        "cells": backed_data.n_obs,
        "genes": backed_data.n_vars,
        "missing_required_obs": ", ".join(missing_columns) or "none",
    })
    backed_data.file.close()

overview_df = pd.DataFrame(dataset_overview)
display(overview_df)
problems = overview_df[overview_df["missing_required_obs"].ne("none")]
if not problems.empty:
    raise KeyError("Some datasets are missing required obs fields; see the table above.")
print("Data file and required field checks passed.")

## Step 4: Train UniCell (Optional)

This step follows the training workflow in the original tutorial, with the title corrected to human liver and tissue/species labels added for the current three-task UniCell model.

Training will:

1. Build the Cell Ontology hierarchy;
2. Save the gene order and ontology used for training;
3. Use the evaluation split to select and save the model.

> If a trained checkpoint is already available, keep `RUN_TRAINING=False`.

In [ ]:
if not RUN_TRAINING:
    print("RUN_TRAINING=False: skipping model training and using the existing model in MODEL_DIR.")
else:
    sc_train = scDataset(
        data_path=str(TRAIN_DATA),
        cell_type_key=CELL_TYPE_KEY,
        tissue_key=TISSUE_KEY,
        species_key=SPECIES_KEY,
        trained=True,
        highly_variable_genes=True,
    )
    input_dim = sc_train.adata.shape[1]

    with (MODEL_DIR / "gene_names.pk").open("wb") as file:
        pickle.dump(sc_train.adata.var_names, file)
    sc_train.ontograph.pickle(str(MODEL_DIR))

    trainer = UnicellTrainer(
        sc_train,
        input_type="expr",
        input_dim=input_dim,
        output_dim=64,
        batch_size=TRAIN_BATCH_SIZE,
        learning_rate=0.001,
        num_epochs=NUM_EPOCHS,
        beta=0.1,
        device=device,
        global_layer=128,
        local_layer=64,
        hidden_layer_dropout=0.1,
        ckpt_dir=str(MODEL_DIR),
    )

    sc_eval = scDataset(
        data_path=str(EVAL_DATA),
        cell_type_key=CELL_TYPE_KEY,
        tissue_key=TISSUE_KEY,
        species_key=SPECIES_KEY,
        trained=False,
        highly_variable_genes=False,
    )
    missing_eval_genes = sc_train.adata.var_names.difference(sc_eval.adata.var_names)
    if len(missing_eval_genes):
        raise ValueError(
            f"The evaluation data are missing {len(missing_eval_genes)} training genes; "
            "align the genes consistently before proceeding."
        )
    sc_eval.adata = sc_eval.adata[:, sc_train.adata.var_names]
    sc_eval.ontograph = sc_train.ontograph
    sc_eval.cell_type_index = sc_eval.get_cell_type_index()

    # Evaluation labels must use the class indices from the training set.
    eval_tissues = sc_eval.adata.obs[TISSUE_KEY].astype(str)
    eval_species = sc_eval.adata.obs[SPECIES_KEY].astype(str)
    unseen_tissues = sorted(set(eval_tissues) - set(sc_train.tissue_label_dict))
    unseen_species = sorted(set(eval_species) - set(sc_train.species_label_dict))
    if unseen_tissues or unseen_species:
        raise ValueError(
            f"The evaluation data contain labels not seen in the training set: "
            f"tissue={unseen_tissues}, species={unseen_species}"
        )
    sc_eval.tissue_label_dict = sc_train.tissue_label_dict
    sc_eval.species_label_dict = sc_train.species_label_dict
    sc_eval.tissue_index = np.array(
        [sc_train.tissue_label_dict[value] for value in eval_tissues], dtype=np.int64
    )
    sc_eval.species_index = np.array(
        [sc_train.species_label_dict[value] for value in eval_species], dtype=np.int64
    )

    trainer.train(scdata_test=sc_eval)
    print(f"Training completed; model saved to: {MODEL_DIR}")

## Step 5: Predict on the Test Set

The current interface requires the test AnnData to be read first, after which both `adata` and the original file path are passed to `unicell_predict`. This tutorial does not use a tissue-specific safe list, so it demonstrates predictions over the full training-class space.

In [ ]:
required_model_files = [
    "unicell_v1.best.pth", "celltype_dict.pk", "tissue_dict.pk",
    "species_dict.pk", "gene_names.pk",
]
missing_model_files = [
    filename for filename in required_model_files
    if not (MODEL_DIR / filename).is_file()
]
if missing_model_files:
    raise FileNotFoundError(
        f"MODEL_DIR is missing model files: {missing_model_files}. "
        "Run the training step first, or point MODEL_DIR to an existing checkpoint."
    )

adata_test = ad.read_h5ad(TEST_DATA)
sc_dataset = unicell_predict(
    adata=adata_test,
    filepath=str(TEST_DATA),
    ckpt_dir=str(MODEL_DIR),
    batch_size=PREDICT_BATCH_SIZE,
    device=device,
    cell_type_key=CELL_TYPE_KEY,
    tissue_key=TISSUE_KEY,
    species_key=SPECIES_KEY,
    compute_metrics=False,
    safe_list_path=None,
)

result_adata = sc_dataset.adata
result_adata.write_h5ad(ANNOTATED_TEST_OUTPUT)
print(f"Predictions completed and saved to: {ANNOTATED_TEST_OUTPUT}")
display(result_adata.obs[[
    "predicted_cell_type_ontology_id",
    "predicted_tissue",
    "predicted_species",
]].head(10))

## Step 6: Evaluate UniCell

If the test set contains ground-truth Cell Ontology IDs, compute Accuracy, Macro-F1, and Micro-F1.

In [ ]:
if CELL_TYPE_KEY not in result_adata.obs.columns:
    print(f"Ground-truth label column '{CELL_TYPE_KEY}' is missing from the test set; skipping evaluation.")
    unicell_metrics = None
else:
    valid = (
        result_adata.obs[CELL_TYPE_KEY].notna()
        & result_adata.obs["predicted_cell_type_ontology_id"].notna()
    )
    y_true_ids = result_adata.obs.loc[valid, CELL_TYPE_KEY].astype(str).tolist()
    y_pred_ids = (
        result_adata.obs.loc[valid, "predicted_cell_type_ontology_id"]
        .astype(str).tolist()
    )
    unicell_metrics = compute_metrics(y_true_ids, y_pred_ids)
    print(f"Number of valid cells for evaluation: {len(y_true_ids):,}")
    display(pd.DataFrame([unicell_metrics], index=["UniCell"]))

## Step 7: Load Predictions from Other Methods (Optional)

The baseline `.h5ad` file's `obs` must contain:

- `celltypist_predicted_cell_type`
- `scanvi_predicted_cell_type`
- `onclass_predicted_cell_type`
- `singler_predicted_cell_type`

CellTypist, scANVI, and SingleR are expected to store cell-type names; OnClass may store Cell Ontology IDs. The code aligns cells by cell ID and does not depend on the row order of the two files.

In [ ]:
baseline_columns = {
    "celltypist": "celltypist_predicted_cell_type",
    "scanvi": "scanvi_predicted_cell_type",
    "onclass": "onclass_predicted_cell_type",
    "singler": "singler_predicted_cell_type",
}

if not BASELINE_PREDICTIONS.is_file():
    print(f"Baseline prediction file not found: {BASELINE_PREDICTIONS}")
    print("Steps 7-9 can be skipped; UniCell evaluation is unaffected.")
    benchmark_obs = None
else:
    baseline_adata = ad.read_h5ad(BASELINE_PREDICTIONS)
    missing_columns = [
        column for column in baseline_columns.values()
        if column not in baseline_adata.obs.columns
    ]
    if missing_columns:
        raise KeyError(f"Missing fields in the baseline file: {missing_columns}")
    missing_cells = result_adata.obs_names.difference(baseline_adata.obs_names)
    if len(missing_cells):
        raise ValueError(f"The baseline file is missing {len(missing_cells)} test cells.")

    benchmark_obs = result_adata.obs.copy()
    aligned_baseline_obs = baseline_adata.obs.reindex(result_adata.obs_names)
    id_to_name = sc_dataset.ontograph.id2name

    def ontology_id_to_name(value):
        value = str(value)
        return str(id_to_name.get(value, value))

    benchmark_obs["true_cell_type"] = benchmark_obs[CELL_TYPE_KEY].map(ontology_id_to_name)
    benchmark_obs["unicell"] = benchmark_obs["predicted_cell_type_ontology_id"].map(ontology_id_to_name)
    benchmark_obs["celltypist"] = aligned_baseline_obs[baseline_columns["celltypist"]].astype(str)
    benchmark_obs["scanvi"] = aligned_baseline_obs[baseline_columns["scanvi"]].astype(str)
    benchmark_obs["onclass"] = aligned_baseline_obs[baseline_columns["onclass"]].map(ontology_id_to_name)
    benchmark_obs["singler"] = aligned_baseline_obs[baseline_columns["singler"]].astype(str)
    display(benchmark_obs[["true_cell_type", *baseline_columns.keys(), "unicell"]].head())

## Step 8: Compute Consistent Metrics for All Five Methods

All methods are compared using cell-type names and are evaluated with Macro-F1, Accuracy, Macro Precision, and Macro Recall. Invalid labels such as `unknown` and missing values are filtered separately for each method.

In [ ]:
UNKNOWN_LABELS = {"", "unknown", "nan", "None", "NA", "N/A"}

def benchmark_method(table, prediction_column):
    true_values = table["true_cell_type"].astype(str)
    predicted_values = table[prediction_column].astype(str)
    valid = ~true_values.isin(UNKNOWN_LABELS) & ~predicted_values.isin(UNKNOWN_LABELS)
    y_true = true_values.loc[valid]
    y_pred = predicted_values.loc[valid]
    return {
        "method": prediction_column,
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "n": int(valid.sum()),
    }

if benchmark_obs is None:
    eval_df = None
    print("No baseline prediction file is available; skipping metrics across methods.")
else:
    method_order = ["unicell", "celltypist", "scanvi", "onclass", "singler"]
    eval_df = pd.DataFrame([benchmark_method(benchmark_obs, method) for method in method_order])
    display(eval_df.style.format({
        "macro_f1": "{:.4f}", "accuracy": "{:.4f}",
        "precision": "{:.4f}", "recall": "{:.4f}",
    }))

## Step 9: Visualize the Evaluation Metrics

The 3D plot compares the Macro-F1, Accuracy, Macro Precision, and Macro Recall of UniCell, CellTypist, scANVI, OnClass, and SingleR.

In [ ]:
if eval_df is None:
    print("eval_df is unavailable; skipping plotting.")
else:
    metric_columns = ["macro_f1", "accuracy", "precision", "recall"]
    metric_labels = ["Macro-F1", "Accuracy", "Precision", "Recall"]
    selected_values = eval_df[metric_columns].to_numpy(dtype=float)
    min_value = max(0.0, float(np.floor(selected_values.min() * 100) / 100 - 0.01))
    max_value = min(1.0, float(np.ceil(selected_values.max() * 100) / 100 + 0.01))
    if min_value == max_value:
        min_value = max(0.0, min_value - 0.05)
        max_value = min(1.0, max_value + 0.05)

    methods = np.arange(len(metric_labels))
    dataset_names = eval_df["method"].values
    datasets = np.arange(len(dataset_names))
    X, Y = np.meshgrid(methods, datasets)
    color_map = dict(zip(dataset_names, BENCHMARK_COLORS))

    fig = plt.figure(figsize=(7, 5.5))
    ax = fig.add_subplot(111, projection="3d")
    for row_index, dataset_name in enumerate(dataset_names):
        color = color_map.get(dataset_name, "gray")
        ax.plot(X[row_index], Y[row_index], selected_values[row_index], "-o", color=color, markersize=5)
        for metric_index in range(selected_values.shape[1] - 1):
            vertices = [[
                [X[row_index, metric_index], Y[row_index, metric_index], min_value],
                [X[row_index, metric_index + 1], Y[row_index, metric_index + 1], min_value],
                [X[row_index, metric_index + 1], Y[row_index, metric_index + 1], selected_values[row_index, metric_index + 1]],
                [X[row_index, metric_index], Y[row_index, metric_index], selected_values[row_index, metric_index]],
            ]]
            ax.add_collection3d(Poly3DCollection(vertices, color=color, alpha=0.3))
        for metric_index, value in enumerate(selected_values[row_index]):
            ax.text(X[row_index, metric_index], Y[row_index, metric_index], value, f"{value:.2f}", ha="center")

    for metric_index in range(selected_values.shape[1]):
        ax.plot(
            X[:, metric_index], Y[:, metric_index], selected_values[:, metric_index],
            "--", color="gray", linewidth=0.5,
        )

    ax.set_zlabel("Evaluation Metrics")
    ax.set_xticks(range(len(metric_labels)))
    ax.set_xticklabels(metric_labels)
    ax.set_yticks(range(len(dataset_names)))
    ax.set_yticklabels(dataset_names)
    ax.set_zlim(min_value, max_value)
    z_ticks = np.linspace(min_value, max_value, 5)
    ax.set_zticks(z_ticks)
    ax.set_zticklabels([f"{tick:.2f}" for tick in z_ticks])
    ax.view_init(elev=30, azim=230)
    ax.set_title("Human liver")
    ax.grid(False)
    plt.tight_layout()
    plt.savefig(METRIC_FIGURE_OUTPUT, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Figure saved to: {METRIC_FIGURE_OUTPUT}")

## Example Figure from the Original Tutorial

The following human liver metric comparison figure is provided by the original Markdown tutorial:

![Evaluation Metrics Comparison](docs/TUTORIALS/01_hsa_liver_metric.png)

> This image shows the static results from the original tutorial. The image generated in the previous step is based on the `eval_df` produced by the current run.

## Outputs

The main outputs include:

- `models/hsa_liver/`: Model checkpoints, the training gene list, class dictionaries, and ontology files.
- `results/hsa_liver_unicell_annotated.h5ad`: The test set with UniCell predicted labels and embeddings.
- `results/hsa_liver_metric.png`: The five-method metric comparison plot generated by the current run.

Common fields in the annotated AnnData:

| Location | Field | Meaning |
|---|---|---|
| `obs` | `predicted_cell_type_ontology_id` | Predicted Cell Ontology ID |
| `obs` | `predicted_tissue` | Predicted tissue |
| `obs` | `predicted_species` | Predicted species |
| `obsm` | `unicell_emb` | UniCell cell representation |
| `obsm` | `cls_emb` | Raw cell-type classification scores |
| `uns` | `cls_cell_type` | Classes corresponding to the cell-type score columns |